# Modular meta-agent: control state vs working data

Worktree-local review notebook. Implementation lives in `src/playbook/`.
This notebook imports that code, renders the graphs, and walks the current
classify → discover → recommend flow.

Behavioral baseline is unchanged from the extracted modular agent:
Jaccard intent match, action-sequence LCS subflow match, greedy clustering,
stub pathway drafts, and `simulate_hitl`.

```text
notebook / UI
     ↓ imports
compact LangGraph application code
     ↓
stable callable agent boundaries
```

## X-ray 0 — system context

```mermaid
flowchart LR
    TS[Task Source] <--> MA[Meta Agent]
    MA <--> KB[KB]
    MA --> LS[LangSmith traces]
```

## X-ray 1 — top-level agent workflow

```mermaid
flowchart TD
    C[Establish Cohort] --> CL[Classify]
    CL --> D[Discover]
    D --> R[Recommend]
    R --> S[Summarize]
```

## X-ray 2 — shared execution pattern

Every subgraph follows the same loop. Task objects live inside the subgraph only.

```mermaid
flowchart LR
    Q[Query store] --> P[Process]
    P --> W[Persist]
    W --> U[Summarize into graph state]
```

After a KB mutation, the next classifier **re-queries the store**. It does not
reuse an in-memory leftover list of unresolved tasks.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

from IPython.display import JSON, Markdown, display

from playbook import (
    build_classify_graph,
    build_cohort_graph,
    build_discover_graph,
    build_intent_discovery_graph,
    build_intent_graph,
    build_meta_graph,
    build_pathway_graph,
    build_subflow_discovery_graph,
    build_subflow_graph,
    configure_runtime,
    empty_state,
    invoke_named,
    invoke_week,
    load_playbook,
    node_summarize_run,
    render_mermaid,
    retrieve_guidance,
)
from playbook import runtime as rt
from playbook.tools import score_intent_similarity, score_subflow_similarity

ROOT = Path(".").resolve()
playbook, store = configure_runtime(data_dir=ROOT / "scratch_data")

print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT:", os.environ.get("LANGSMITH_PROJECT"))
print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("kb_version", playbook.version)
print("intent docs:", {i: playbook.intent_doc(i) for i in playbook.intent_ids()})
print("loaded tasks:", store.fetchall("SELECT task_id, hidden_flow, hidden_subflow, success FROM tasks"))

LANGSMITH_TRACING: true
LANGSMITH_PROJECT: fresh-skills
LANGSMITH_API_KEY set: True
kb_version 1
intent docs: {'account_access': 'Account Access account_access username, password, two-factor authentication, and other login / account-access problems including lockouts Recover Username Recover Password recover_username recover_password'}
loaded tasks: [{'task_id': 'u1', 'hidden_flow': 'account_access', 'hidden_subflow': 'recover_username', 'success': 1}, {'task_id': 'p1', 'hidden_flow': 'account_access', 'hidden_subflow': 'recover_password', 'success': 1}, {'task_id': 't1', 'hidden_flow': 'account_access', 'hidden_subflow': 'reset_2fa', 'success': 1}, {'task_id': 't2', 'hidden_flow': 'account_access', 'hidden_subflow': 'reset_2fa', 'success': 1}, {'task_id': 't3', 'hidden_flow': 'account_access', 'hidden_subflow': 'reset_2fa', 'success': 1}, {'task_id': 'l1', 'hidden_flow': 'account_access', 'hidden_subflow': 'account_locked_mixed', 'success': 0}, {'task_id': 'l2', 'hidden_flow': 'accoun

In [2]:
def show(obj: Any, title: str = "") -> None:
    if title:
        display(Markdown(f"**{title}**"))
    display(Markdown(f"```json\n{json.dumps(obj, indent=2, default=str)}\n```"))


def show_state(state, keys: list[str] | None = None, title: str = "state") -> None:
    picked = keys or [
        "run_id",
        "kb_version",
        "current_stage",
        "intent_summary",
        "subflow_summary",
        "discovery_summary",
        "recommendation_summary",
        "pending_proposal_ids",
        "approved_change_ids",
    ]
    show({k: state.get(k) for k in picked}, title)


def show_rows(rows: list[dict[str, Any]], title: str, keep: list[str] | None = None) -> None:
    trimmed = []
    for row in rows:
        item = {k: row[k] for k in keep if k in row} if keep else dict(row)
        trimmed.append(item)
    show(trimmed, title)


def show_mermaid(compiled, *, xray: bool | int = False, title: str = "") -> None:
    source = render_mermaid(compiled, xray=xray)
    if title:
        display(Markdown(f"### {title}"))
    display(Markdown(f"```mermaid\n{source}\n```"))

Hidden `scenario.flow` / `scenario.subflow` are **offline labels** for evaluation
in this notebook. Classifiers and discovery never read them.

In [3]:
JSON(
    {
        "version": playbook.version,
        "flows": playbook.intent_ids(),
        "subflows": playbook.ontology["intents"]["subflows"],
        "kb": playbook.kb,
    }
)

<IPython.core.display.JSON object>

## Canonical loader / retriever / tools

`load_playbook` and `retrieve_guidance` are independent of the graph. The
LangChain tools still use the bound runtime store.

In [4]:
seed = load_playbook(ROOT / "scratch_data")
print(retrieve_guidance("I forgot my username and cannot log into my account.", seed))
print("sanity scores (intent=account_access)")
for cid in ["u1", "p1", "t1", "l1", "s1", "x1"]:
    intent = score_intent_similarity.invoke({"conversation_id": cid, "intent_id": "account_access"})
    username = score_subflow_similarity.invoke({"conversation_id": cid, "subflow_id": "recover_username"})
    password = score_subflow_similarity.invoke({"conversation_id": cid, "subflow_id": "recover_password"})
    print(f"  {cid:3}  intent={intent:.3f}  username={username:.3f}  password={password:.3f}")

[{'intent_id': 'account_access', 'score': 0.15, 'doc': 'Account Access account_access username, password, two-factor authentication, and other login / account-access problems including lockouts Recover Username Recover Password recover_username recover_password', 'subflows': ['recover_username', 'recover_password']}]
sanity scores (intent=account_access)


  u1   intent=0.091  username=1.000  password=0.333


  p1   intent=0.078  username=0.333  password=1.000
  t1   intent=0.107  username=0.333  password=0.667
  l1   intent=0.077  username=0.000  password=0.000
  s1   intent=0.029  username=0.200  password=0.200
  x1   intent=0.000  username=0.000  password=0.000


## Compiled graphs

Display graphs use compiled subgraphs; the run graph calls `invoke`.

In [5]:
cohort_graph = build_cohort_graph()
intent_graph = build_intent_graph()
subflow_graph = build_subflow_graph()
intent_discovery_graph = build_intent_discovery_graph()
subflow_discovery_graph = build_subflow_discovery_graph()
pathway_graph = build_pathway_graph()
classify_display = build_classify_graph(True)
discover_display = build_discover_graph(True)
meta_display = build_meta_graph(True)
meta_graph = build_meta_graph(False)

show_mermaid(meta_display, title="Top-level meta-agent")
show_mermaid(classify_display, title="Classify")
show_mermaid(discover_display, title="Discover (includes reclassify after KB mutation)")
show_mermaid(intent_graph, title="classify_intents")
show_mermaid(subflow_graph, title="classify_subflows")
show_mermaid(intent_discovery_graph, title="discover_intents")
show_mermaid(subflow_discovery_graph, title="discover_subflows")
show_mermaid(pathway_graph, title="recommend_pathway")
show_mermaid(meta_display, xray=1, title="Top-level xray=1")

### Top-level meta-agent

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	establish_cohort(establish_cohort<hr/><small><em>agent = meta
phase = query</em></small>)
	classify(classify<hr/><small><em>agent = meta
phase = process</em></small>)
	discover(discover<hr/><small><em>agent = meta
phase = process</em></small>)
	recommend(recommend<hr/><small><em>agent = pathway
phase = process</em></small>)
	summarize(summarize<hr/><small><em>agent = meta
phase = summarize</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> establish_cohort;
	classify --> discover;
	discover --> recommend;
	establish_cohort --> classify;
	recommend --> summarize;
	summarize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

### Classify

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify_intents(classify_intents<hr/><small><em>agent = intent
phase = process</em></small>)
	classify_subflows(classify_subflows<hr/><small><em>agent = subflow
phase = process</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify_intents;
	classify_intents --> classify_subflows;
	classify_subflows --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

### Discover (includes reclassify after KB mutation)

...

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	query(query<hr/><small><em>agent = subflow_discovery
phase = query</em></small>)
	discover(discover<hr/><small><em>agent = subflow_discovery
phase = process</em></small>)
	validate(validate<hr/><small><em>agent = subflow_discovery
phase = process</em></small>)
	persist_proposal(persist_proposal<hr/><small><em>agent = subflow_discovery
phase = persist</em></small>)
	hitl(hitl<hr/><small><em>agent = subflow_discovery
phase = process</em></small>)
	persist_kb(persist_kb<hr/><small><em>agent = subflow_discovery
phase = persist</em></small>)
	summarize(summarize<hr/><small><em>agent = subflow_discovery
phase = summarize</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> query;
	discover --> validate;
	hitl --> persist_kb;
	persist_kb --> summarize;
	persist_proposal --> hitl;
	query --> discover;
	validate --> persist_proposal;
	summarize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

### recommend_pathway

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	query(query<hr/><small><em>agent = pathway
phase = query</em></small>)
	analyze(analyze<hr/><small><em>agent = pathway
phase = process</em></small>)
	recommend(recommend<hr/><small><em>agent = pathway
phase = process</em></small>)
	evaluate(evaluate<hr/><small><em>agent = pathway
phase = process</em></small>)
	persist(persist<hr/><small><em>agent = pathway
phase = persist</em></small>)
	summarize(summarize<hr/><small><em>agent = pathway
phase = summarize</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> query;
	analyze --> recommend;
	evaluate --> persist;
	persist --> summarize;
	query --> analyze;
	recommend --> evaluate;
	summarize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

### Top-level xray=1

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	establish_cohort(establish_cohort<hr/><small><em>agent = meta
phase = query</em></small>)
	recommend(recommend<hr/><small><em>agent = pathway
phase = process</em></small>)
	summarize(summarize<hr/><small><em>agent = meta
phase = summarize</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> establish_cohort;
	classify\3aclassify_subflows --> discover\3adiscover_intents;
	discover\3areclassify_subflows --> recommend;
	establish_cohort --> classify\3aclassify_intents;
	recommend --> summarize;
	summarize --> __end__;
	subgraph classify
	classify\3aclassify_intents(classify_intents<hr/><small><em>agent = intent
phase = process</em></small>)
	classify\3aclassify_subflows(classify_subflows<hr/><small><em>agent = subflow
phase = process</em></small>)
	classify\3aclassify_intents --> classify\3aclassify_subflows;
	end
	subgraph discover
	discover\3adiscover_intents(discover_intents<hr/><small><em>agent = intent_discovery
phase = process</em></small>)
	discover\3areclassify_intents(reclassify_intents<hr/><small><em>agent = intent
phase = process</em></small>)
	discover\3adiscover_subflows(discover_subflows<hr/><small><em>agent = subflow_discovery
phase = process</em></small>)
	discover\3areclassify_subflows(reclassify_subflows<hr/><small><em>agent = subflow
phase = process</em></small>)
	discover\3adiscover_intents --> discover\3areclassify_intents;
	discover\3adiscover_subflows --> discover\3areclassify_subflows;
	discover\3areclassify_intents --> discover\3adiscover_subflows;
	end
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Evaluation helper (offline labels only)

Classifiers never see `hidden_flow` / `hidden_subflow`. This is notebook-only
coverage / correctness visibility.

In [6]:
def eval_intents(run_id: str) -> list[dict[str, Any]]:
    latest = store.latest_intents(run_id)
    rows = []
    for task in store.get_tasks(store.cohort_ids(run_id)):
        pred = latest.get(task["task_id"])
        pred_id = pred["intent_id"] if pred else None
        truth = task["hidden_flow"]
        if pred_id is None or pred_id == "unknown":
            outcome = "unresolved"
        elif truth in {None, "outlier"}:
            outcome = "unresolved" if pred_id == "unknown" else "predicted_without_gt"
        elif pred_id == truth:
            outcome = "correct"
        else:
            outcome = "incorrect"
        rows.append(
            {
                "task_id": task["task_id"],
                "predicted": pred_id,
                "hidden": truth,
                "confidence": pred["confidence"] if pred else None,
                "outcome": outcome,
            }
        )
    return rows


def eval_subflows(run_id: str) -> list[dict[str, Any]]:
    latest = store.latest_subflows(run_id)
    rows = []
    for task in store.get_tasks(store.cohort_ids(run_id)):
        pred = latest.get(task["task_id"])
        pred_id = pred["subflow_id"] if pred else None
        truth = task["hidden_subflow"]
        if pred_id is None or pred_id == "unknown":
            outcome = "unresolved"
        elif pred_id == truth:
            outcome = "correct"
        else:
            outcome = "incorrect"
        rows.append(
            {
                "task_id": task["task_id"],
                "predicted": pred_id,
                "hidden": truth,
                "confidence": pred["confidence"] if pred else None,
                "outcome": outcome,
            }
        )
    return rows


def count_outcomes(rows: list[dict[str, Any]]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for row in rows:
        counts[row["outcome"]] = counts.get(row["outcome"], 0) + 1
    return counts

---
# Walkthrough

Expected operational outcomes:

| ids | hidden | hoped-for path |
| --- | --- | --- |
| `u1`, `p1` | known subflows | classify existing, no structural change |
| `t1–t3` | `reset_2fa` | existing intent, discover + recommend subflow |
| `l1–l3` | locked / mixed | coherent cluster, no successful pattern, decline |
| `s1–s3` | shipping missing | discover intent, reclassify, discover subflow, recommend |
| `x1` | store window | outlier / monitor |

### 1. Inspect current KB

In [7]:
show(
    {
        "version": playbook.version,
        "flows": playbook.intent_ids(),
        "subflows": playbook.ontology["intents"]["subflows"],
        "kb": playbook.kb,
    },
    "seed KB",
)

**seed KB**

```json
{
  "version": 1,
  "flows": [
    "account_access"
  ],
  "subflows": {
    "account_access": [
      "recover_username",
      "recover_password"
    ]
  },
  "kb": {
    "recover_username": [
      "pull-up-account",
      "verify-identity"
    ],
    "recover_password": [
      "pull-up-account",
      "enter-details",
      "make-password"
    ]
  }
}
```

### 2. Establish weekly cohort

In [8]:
state = empty_state(
    run_id="week-2026-09-01",
    cohort_query={
        "source": "scratch_data/incoming_conversations.json",
        "window": "demo-week",
        "as_of": "2026-09-01",
    },
)
state = invoke_named(cohort_graph, state, agent="meta")
show_state(state, keys=["run_id", "cohort_query", "kb_version", "current_stage"], title="after cohort")
show_rows(
    store.fetchall("SELECT run_id, task_id FROM cohort ORDER BY task_id"),
    "cohort mapping (run_id | task_id)",
)

**after cohort**

```json
{
  "run_id": "week-2026-09-01",
  "cohort_query": {
    "source": "scratch_data/incoming_conversations.json",
    "window": "demo-week",
    "as_of": "2026-09-01"
  },
  "kb_version": 1,
  "current_stage": "cohort.summarize"
}
```

**cohort mapping (run_id | task_id)**

```json
[
  {
    "run_id": "week-2026-09-01",
    "task_id": "l1"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "l2"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "l3"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "p1"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "s1"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "s2"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "s3"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "t1"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "t2"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "t3"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "u1"
  },
  {
    "run_id": "week-2026-09-01",
    "task_id": "x1"
  }
]
```

### 3–4. Intent classification + persisted records

In [9]:
state = invoke_named(intent_graph, state, agent="intent")
show_state(state, keys=["kb_version", "current_stage", "intent_summary"], title="after intent classification")
show_rows(
    store.fetchall(
        """
        SELECT task_id, intent_id, confidence, method, kb_version
        FROM intent_labels WHERE run_id = ? ORDER BY id
        """,
        (state["run_id"],),
    ),
    "persisted intent labels",
)
intent_eval = eval_intents(state["run_id"])
show_rows(intent_eval, "intent eval vs hidden labels")
print("intent outcomes:", count_outcomes(intent_eval))

**after intent classification**

```json
{
  "kb_version": 1,
  "current_stage": "intent.summarize",
  "intent_summary": {
    "total": 12,
    "processed": 12,
    "classified": 8,
    "unknown": 4,
    "low_confidence": 3,
    "by_intent": {
      "account_access": 8,
      "unknown": 4
    },
    "still_unresolved": 4
  }
}
```

**persisted intent labels**

```json
[
  {
    "task_id": "l1",
    "intent_id": "account_access",
    "confidence": 0.077,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "l2",
    "intent_id": "account_access",
    "confidence": 0.089,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "l3",
    "intent_id": "account_access",
    "confidence": 0.095,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "p1",
    "intent_id": "account_access",
    "confidence": 0.078,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "s1",
    "intent_id": "unknown",
    "confidence": 0.029,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "s2",
    "intent_id": "unknown",
    "confidence": 0.039,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "s3",
    "intent_id": "unknown",
    "confidence": 0.036,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "t1",
    "intent_id": "account_access",
    "confidence": 0.107,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "t2",
    "intent_id": "account_access",
    "confidence": 0.122,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "t3",
    "intent_id": "account_access",
    "confidence": 0.093,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "u1",
    "intent_id": "account_access",
    "confidence": 0.091,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  },
  {
    "task_id": "x1",
    "intent_id": "unknown",
    "confidence": 0.0,
    "method": "jaccard_intent_doc_v1",
    "kb_version": 1
  }
]
```

**intent eval vs hidden labels**

```json
[
  {
    "task_id": "l1",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.077,
    "outcome": "correct"
  },
  {
    "task_id": "l2",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.089,
    "outcome": "correct"
  },
  {
    "task_id": "l3",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.095,
    "outcome": "correct"
  },
  {
    "task_id": "p1",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.078,
    "outcome": "correct"
  },
  {
    "task_id": "s1",
    "predicted": "unknown",
    "hidden": "shipping_issue",
    "confidence": 0.029,
    "outcome": "unresolved"
  },
  {
    "task_id": "s2",
    "predicted": "unknown",
    "hidden": "shipping_issue",
    "confidence": 0.039,
    "outcome": "unresolved"
  },
  {
    "task_id": "s3",
    "predicted": "unknown",
    "hidden": "shipping_issue",
    "confidence": 0.036,
    "outcome": "unresolved"
  },
  {
    "task_id": "t1",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.107,
    "outcome": "correct"
  },
  {
    "task_id": "t2",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.122,
    "outcome": "correct"
  },
  {
    "task_id": "t3",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.093,
    "outcome": "correct"
  },
  {
    "task_id": "u1",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.091,
    "outcome": "correct"
  },
  {
    "task_id": "x1",
    "predicted": "unknown",
    "hidden": "outlier",
    "confidence": 0.0,
    "outcome": "unresolved"
  }
]
```

intent outcomes: {'correct': 8, 'unresolved': 4}


### 5–6. Subflow classification + summary

In [10]:
state = invoke_named(subflow_graph, state, agent="subflow")
show_state(state, keys=["kb_version", "current_stage", "subflow_summary"], title="after subflow classification")
show_rows(
    store.fetchall(
        """
        SELECT task_id, intent_id, subflow_id, confidence, method, kb_version
        FROM subflow_labels WHERE run_id = ? ORDER BY id
        """,
        (state["run_id"],),
    ),
    "persisted subflow labels",
)
print("subflow outcomes:", count_outcomes(eval_subflows(state["run_id"])))

**after subflow classification**

```json
{
  "kb_version": 1,
  "current_stage": "subflow.summarize",
  "subflow_summary": {
    "processed": 8,
    "classified": 2,
    "unknown": 6,
    "low_confidence": 0,
    "by_intent": {
      "account_access": {
        "unknown": 6,
        "recover_password": 1,
        "recover_username": 1
      }
    },
    "still_unresolved": 6
  }
}
```

**persisted subflow labels**

```json
[
  {
    "task_id": "l1",
    "intent_id": "account_access",
    "subflow_id": "unknown",
    "confidence": 0.0,
    "method": "lcs_kb_actions_v1",
    "kb_version": 1
  },
  {
    "task_id": "l2",
    "intent_id": "account_access",
    "subflow_id": "unknown",
    "confidence": 0.0,
    "method": "lcs_kb_actions_v1",
    "kb_version": 1
  },
  {
    "task_id": "l3",
    "intent_id": "account_access",
    "subflow_id": "unknown",
    "confidence": 0.0,
    "method": "lcs_kb_actions_v1",
    "kb_version": 1
  },
  {
    "task_id": "p1",
    "intent_id": "account_access",
    "subflow_id": "recover_password",
    "confidence": 1.0,
    "method": "lcs_kb_actions_v1",
    "kb_version": 1
  },
  {
    "task_id": "t1",
    "intent_id": "account_access",
    "subflow_id": "unknown",
    "confidence": 0.667,
    "method": "lcs_kb_actions_v1",
    "kb_version": 1
  },
  {
    "task_id": "t2",
    "intent_id": "account_access",
    "subflow_id": "unknown",
    "confidence": 0.667,
    "method": "lcs_kb_actions_v1",
    "kb_version": 1
  },
  {
    "task_id": "t3",
    "intent_id": "account_access",
    "subflow_id": "unknown",
    "confidence": 0.667,
    "method": "lcs_kb_actions_v1",
    "kb_version": 1
  },
  {
    "task_id": "u1",
    "intent_id": "account_access",
    "subflow_id": "recover_username",
    "confidence": 1.0,
    "method": "lcs_kb_actions_v1",
    "kb_version": 1
  }
]
```

subflow outcomes: {'unresolved': 10, 'correct': 2}


### 7–11. Discover intents, simulated HITL, persist KB, reclassify

Unresolved intents are **re-queried from the store**, not passed as a Python list.

In [11]:
print("unresolved intents before discovery:", store.unresolved_intent_ids(state["run_id"]))
state = invoke_named(intent_discovery_graph, state, agent="intent_discovery")
show_state(
    state,
    keys=["kb_version", "current_stage", "discovery_summary", "pending_proposal_ids", "approved_change_ids"],
    title="after intent discovery",
)
show_rows(
    [
        {
            "proposal_id": p["proposal_id"],
            "type": p["proposal_type"],
            "candidate": p["candidate"],
            "tasks": p["supporting_task_ids"],
            "metrics": p["metrics"],
            "decision": p["review_decision"],
            "note": p["review_note"],
            "kb_version": p["resulting_kb_version"],
        }
        for p in store.list_proposals(state["run_id"])
    ],
    "intent-stage proposals + HITL",
)
print("KB intents now:", playbook.intent_ids())
print("unresolved intents after KB persist (before reclassify):", store.unresolved_intent_ids(state["run_id"]))
state = invoke_named(intent_graph, state, agent="intent")
show_state(state, keys=["intent_summary"], title="after reclassify intents")
show_rows(eval_intents(state["run_id"]), "intent eval after reclassify")
print("intent outcomes after reclassify:", count_outcomes(eval_intents(state["run_id"])))
print("unresolved intents after reclassify:", store.unresolved_intent_ids(state["run_id"]))

unresolved intents before discovery: ['s1', 's2', 's3', 'x1']


**after intent discovery**

```json
{
  "kb_version": 2,
  "current_stage": "intent_discovery.summarize",
  "discovery_summary": {
    "intents": {
      "unresolved_tasks": 4,
      "candidate_count": 1,
      "outlier_count": 1,
      "approved_count": 1,
      "rejected_count": 1
    }
  },
  "pending_proposal_ids": [
    "prop_6373f611",
    "prop_e9ef9137"
  ],
  "approved_change_ids": [
    "prop_6373f611"
  ]
}
```

**intent-stage proposals + HITL**

```json
[
  {
    "proposal_id": "prop_6373f611",
    "type": "new_intent",
    "candidate": "shipping_issue",
    "tasks": "[\"s1\", \"s2\", \"s3\"]",
    "metrics": "{\"size\": 3, \"mean_jaccard\": 0.383, \"coherent\": true}",
    "decision": "accept",
    "note": "Coherent new intent; enough shipping evidence.",
    "kb_version": 2
  },
  {
    "proposal_id": "prop_e9ef9137",
    "type": "outlier",
    "candidate": null,
    "tasks": "[\"x1\"]",
    "metrics": "{\"size\": 1, \"mean_jaccard\": 0.0, \"coherent\": false}",
    "decision": "decline",
    "note": "Placeholder HITL: monitor / no playbook change.",
    "kb_version": null
  }
]
```

...

**after reclassify intents**

```json
{
  "intent_summary": {
    "total": 12,
    "processed": 1,
    "classified": 0,
    "unknown": 1,
    "low_confidence": 0,
    "by_intent": {
      "account_access": 8,
      "shipping_issue": 3,
      "unknown": 1
    },
    "still_unresolved": 1
  }
}
```

**intent eval after reclassify**

```json
[
  {
    "task_id": "l1",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.077,
    "outcome": "correct"
  },
  {
    "task_id": "l2",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.089,
    "outcome": "correct"
  },
  {
    "task_id": "l3",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.095,
    "outcome": "correct"
  },
  {
    "task_id": "p1",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.078,
    "outcome": "correct"
  },
  {
    "task_id": "s1",
    "predicted": "shipping_issue",
    "hidden": "shipping_issue",
    "confidence": 0.383,
    "outcome": "correct"
  },
  {
    "task_id": "s2",
    "predicted": "shipping_issue",
    "hidden": "shipping_issue",
    "confidence": 0.383,
    "outcome": "correct"
  },
  {
    "task_id": "s3",
    "predicted": "shipping_issue",
    "hidden": "shipping_issue",
    "confidence": 0.383,
    "outcome": "correct"
  },
  {
    "task_id": "t1",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.107,
    "outcome": "correct"
  },
  {
    "task_id": "t2",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.122,
    "outcome": "correct"
  },
  {
    "task_id": "t3",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.093,
    "outcome": "correct"
  },
  {
    "task_id": "u1",
    "predicted": "account_access",
    "hidden": "account_access",
    "confidence": 0.091,
    "outcome": "correct"
  },
  {
    "task_id": "x1",
    "predicted": "unknown",
    "hidden": "outlier",
    "confidence": 0.0,
    "outcome": "unresolved"
  }
]
```

intent outcomes after reclassify: {'correct': 11, 'unresolved': 1}
unresolved intents after reclassify: ['x1']


### 12–15. Discover subflows by intent, HITL, persist, reclassify

In [12]:
print("unresolved subflows before discovery:", store.unresolved_subflow_ids(state["run_id"]))
state = invoke_named(subflow_discovery_graph, state, agent="subflow_discovery")
show_state(
    state,
    keys=["kb_version", "current_stage", "discovery_summary", "approved_change_ids"],
    title="after subflow discovery",
)
show_rows(
    [
        {
            "proposal_id": p["proposal_id"],
            "type": p["proposal_type"],
            "intent": p["parent_intent"],
            "candidate": p["candidate"],
            "tasks": p["supporting_task_ids"],
            "decision": p["review_decision"],
            "note": p["review_note"],
            "kb_version": p["resulting_kb_version"],
        }
        for p in store.list_proposals(state["run_id"])
        if p["proposal_type"] in {"new_subflow", "emerging", "outlier"}
    ],
    "subflow-stage proposals + HITL",
)
print("KB subflows now:", playbook.ontology["intents"]["subflows"])
state = invoke_named(subflow_graph, state, agent="subflow")
show_state(state, keys=["subflow_summary"], title="after reclassify subflows")
show_rows(eval_subflows(state["run_id"]), "subflow eval after reclassify")
print("subflow outcomes:", count_outcomes(eval_subflows(state["run_id"])))

unresolved subflows before discovery: ['l1', 'l2', 'l3', 's1', 's2', 's3', 't1', 't2', 't3']


**after subflow discovery**

```json
{
  "kb_version": 4,
  "current_stage": "subflow_discovery.summarize",
  "discovery_summary": {
    "intents": {
      "unresolved_tasks": 4,
      "candidate_count": 1,
      "outlier_count": 1,
      "approved_count": 1,
      "rejected_count": 1
    },
    "subflows": {
      "unresolved_tasks": 9,
      "candidate_count": 2,
      "emerging_count": 1,
      "approved_count": 2,
      "rejected_count": 1
    }
  },
  "approved_change_ids": [
    "prop_6373f611",
    "prop_77122c73",
    "prop_29f9b0e1"
  ]
}
```

**subflow-stage proposals + HITL**

```json
[
  {
    "proposal_id": "prop_e9ef9137",
    "type": "outlier",
    "intent": null,
    "candidate": null,
    "tasks": "[\"x1\"]",
    "decision": "decline",
    "note": "Placeholder HITL: monitor / no playbook change.",
    "kb_version": null
  },
  {
    "proposal_id": "prop_b64f8e39",
    "type": "emerging",
    "intent": "account_access",
    "candidate": null,
    "tasks": "[\"l1\", \"l2\", \"l3\"]",
    "decision": "decline",
    "note": "Placeholder HITL: monitor / no playbook change.",
    "kb_version": null
  },
  {
    "proposal_id": "prop_77122c73",
    "type": "new_subflow",
    "intent": "account_access",
    "candidate": "reset_2fa",
    "tasks": "[\"t1\", \"t2\", \"t3\"]",
    "decision": "accept",
    "note": "Repeatable 2FA reset pattern.",
    "kb_version": 3
  },
  {
    "proposal_id": "prop_29f9b0e1",
    "type": "new_subflow",
    "intent": "shipping_issue",
    "candidate": "missing",
    "tasks": "[\"s1\", \"s2\", \"s3\"]",
    "decision": "accept",
    "note": "Repeatable missing-package resolution.",
    "kb_version": 4
  }
]
```

...

**after reclassify subflows**

```json
{
  "subflow_summary": {
    "processed": 3,
    "classified": 0,
    "unknown": 3,
    "low_confidence": 0,
    "by_intent": {
      "account_access": {
        "unknown": 3,
        "recover_password": 1,
        "reset_2fa": 3,
        "recover_username": 1
      },
      "shipping_issue": {
        "missing": 3
      }
    },
    "still_unresolved": 3
  }
}
```

**subflow eval after reclassify**

```json
[
  {
    "task_id": "l1",
    "predicted": "unknown",
    "hidden": "account_locked_mixed",
    "confidence": 0.0,
    "outcome": "unresolved"
  },
  {
    "task_id": "l2",
    "predicted": "unknown",
    "hidden": "account_locked_mixed",
    "confidence": 0.0,
    "outcome": "unresolved"
  },
  {
    "task_id": "l3",
    "predicted": "unknown",
    "hidden": "account_locked_mixed",
    "confidence": 0.0,
    "outcome": "unresolved"
  },
  {
    "task_id": "p1",
    "predicted": "recover_password",
    "hidden": "recover_password",
    "confidence": 1.0,
    "outcome": "correct"
  },
  {
    "task_id": "s1",
    "predicted": "missing",
    "hidden": "missing",
    "confidence": 1.0,
    "outcome": "correct"
  },
  {
    "task_id": "s2",
    "predicted": "missing",
    "hidden": "missing",
    "confidence": 1.0,
    "outcome": "correct"
  },
  {
    "task_id": "s3",
    "predicted": "missing",
    "hidden": "missing",
    "confidence": 1.0,
    "outcome": "correct"
  },
  {
    "task_id": "t1",
    "predicted": "reset_2fa",
    "hidden": "reset_2fa",
    "confidence": 1.0,
    "outcome": "correct"
  },
  {
    "task_id": "t2",
    "predicted": "reset_2fa",
    "hidden": "reset_2fa",
    "confidence": 1.0,
    "outcome": "correct"
  },
  {
    "task_id": "t3",
    "predicted": "reset_2fa",
    "hidden": "reset_2fa",
    "confidence": 1.0,
    "outcome": "correct"
  },
  {
    "task_id": "u1",
    "predicted": "recover_username",
    "hidden": "recover_username",
    "confidence": 1.0,
    "outcome": "correct"
  },
  {
    "task_id": "x1",
    "predicted": null,
    "hidden": "store_window_display",
    "confidence": null,
    "outcome": "unresolved"
  }
]
```

subflow outcomes: {'unresolved': 4, 'correct': 8}


### 16–17. Pathway recommendation for newly validated subflows

In [13]:
for proposal in store.list_proposals(state["run_id"], "new_subflow"):
    if proposal["review_decision"] != "accept":
        continue
    print("recommend pathway for", proposal["candidate"], "tasks", proposal["supporting_task_ids"])
    state = invoke_named(
        pathway_graph,
        {**state, "target_subflow": proposal["candidate"]},
        agent="pathway",
    )
    show_state(state, keys=["recommendation_summary", "kb_version"], title=f"after pathway {proposal['candidate']}")

show_rows(
    [
        {
            "rec_id": r["rec_id"],
            "intent_id": r["intent_id"],
            "subflow_id": r["subflow_id"],
            "evaluation": r["evaluation"],
            "kb_draft": r["kb_draft"],
        }
        for r in store.list_recommendations(state["run_id"])
    ],
    "persisted recommendations",
)

recommend pathway for reset_2fa tasks ["t1", "t2", "t3"]


**after pathway reset_2fa**

```json
{
  "recommendation_summary": {
    "items": [
      {
        "subflow_id": "reset_2fa",
        "intent_id": "account_access",
        "supported": true,
        "n_tasks": 3,
        "support": 1.0
      }
    ],
    "recommended": 1
  },
  "kb_version": 5
}
```

recommend pathway for missing tasks ["s1", "s2", "s3"]


**after pathway missing**

```json
{
  "recommendation_summary": {
    "items": [
      {
        "subflow_id": "reset_2fa",
        "intent_id": "account_access",
        "supported": true,
        "n_tasks": 3,
        "support": 1.0
      },
      {
        "subflow_id": "missing",
        "intent_id": "shipping_issue",
        "supported": true,
        "n_tasks": 3,
        "support": 1.0
      }
    ],
    "recommended": 2
  },
  "kb_version": 6
}
```

**persisted recommendations**

```json
[
  {
    "rec_id": "rec_39755ef3",
    "intent_id": "account_access",
    "subflow_id": "reset_2fa",
    "evaluation": "{\"supported\": true, \"successful\": 3, \"support\": 1.0}",
    "kb_draft": "{\"reset_2fa\": [\"pull-up-account\", \"enter-details\", \"send-link\"]}"
  },
  {
    "rec_id": "rec_47521206",
    "intent_id": "shipping_issue",
    "subflow_id": "missing",
    "evaluation": "{\"supported\": true, \"successful\": 3, \"support\": 1.0}",
    "kb_draft": "{\"missing\": [\"pull-up-account\", \"validate-purchase\", \"record-reason\", \"update-order\", \"make-purchase\"]}"
  }
]
```

### 18–19. Final run summary and KB changes

In [14]:
state = {**state, **node_summarize_run(state)}
show(store.get_staging(state["run_id"], "run_summary"), "final run summary")
show(
    {
        "version": playbook.version,
        "flows": playbook.intent_ids(),
        "subflows": playbook.ontology["intents"]["subflows"],
        "kb": playbook.kb,
        "guideline_flow_keys": {k: list(v.get("subflows", {})) for k, v in playbook.guidelines.items()},
    },
    "final KB",
)
show_rows(store.fetchall("SELECT kb_version, change_type, payload FROM kb_log ORDER BY kb_version"), "KB mutation log")
print("still unresolved intents:", store.unresolved_intent_ids(state["run_id"]))
print("still unresolved subflows:", store.unresolved_subflow_ids(state["run_id"]))

**final run summary**

```json
{
  "run_id": "week-2026-09-01",
  "kb_version": 6,
  "cohort_size": 12,
  "intent_summary": {
    "total": 12,
    "processed": 1,
    "classified": 0,
    "unknown": 1,
    "low_confidence": 0,
    "by_intent": {
      "account_access": 8,
      "shipping_issue": 3,
      "unknown": 1
    },
    "still_unresolved": 1
  },
  "subflow_summary": {
    "processed": 3,
    "classified": 0,
    "unknown": 3,
    "low_confidence": 0,
    "by_intent": {
      "account_access": {
        "unknown": 3,
        "recover_password": 1,
        "reset_2fa": 3,
        "recover_username": 1
      },
      "shipping_issue": {
        "missing": 3
      }
    },
    "still_unresolved": 3
  },
  "discovery_summary": {
    "intents": {
      "unresolved_tasks": 4,
      "candidate_count": 1,
      "outlier_count": 1,
      "approved_count": 1,
      "rejected_count": 1
    },
    "subflows": {
      "unresolved_tasks": 9,
      "candidate_count": 2,
      "emerging_count": 1,
      "approved_count": 2,
      "rejected_count": 1
    }
  },
  "recommendation_summary": {
    "items": [
      {
        "subflow_id": "reset_2fa",
        "intent_id": "account_access",
        "supported": true,
        "n_tasks": 3,
        "support": 1.0
      },
      {
        "subflow_id": "missing",
        "intent_id": "shipping_issue",
        "supported": true,
        "n_tasks": 3,
        "support": 1.0
      }
    ],
    "recommended": 2
  },
  "intents_in_kb": [
    "account_access",
    "shipping_issue"
  ],
  "subflows_in_kb": {
    "account_access": [
      "recover_username",
      "recover_password",
      "reset_2fa"
    ],
    "shipping_issue": [
      "missing"
    ]
  },
  "unresolved_intents": [
    "x1"
  ],
  "unresolved_subflows": [
    "l1",
    "l2",
    "l3"
  ]
}
```

**final KB**

```json
{
  "version": 6,
  "flows": [
    "account_access",
    "shipping_issue"
  ],
  "subflows": {
    "account_access": [
      "recover_username",
      "recover_password",
      "reset_2fa"
    ],
    "shipping_issue": [
      "missing"
    ]
  },
  "kb": {
    "recover_username": [
      "pull-up-account",
      "verify-identity"
    ],
    "recover_password": [
      "pull-up-account",
      "enter-details",
      "make-password"
    ],
    "reset_2fa": [
      "pull-up-account",
      "enter-details",
      "send-link"
    ],
    "missing": [
      "pull-up-account",
      "validate-purchase",
      "record-reason",
      "update-order",
      "make-purchase"
    ]
  },
  "guideline_flow_keys": {
    "Account Access": [
      "Recover Username",
      "Recover Password",
      "Reset Two-Factor Auth"
    ],
    "Shipping Issue": [
      "Missing Item"
    ]
  }
}
```

**KB mutation log**

```json
[
  {
    "kb_version": 2,
    "change_type": "add_intent",
    "payload": "{\"intent\": \"shipping_issue\", \"tasks\": [\"s1\", \"s2\", \"s3\"]}"
  },
  {
    "kb_version": 3,
    "change_type": "add_subflow",
    "payload": "{\"intent\": \"account_access\", \"subflow\": \"reset_2fa\", \"tasks\": [\"t1\", \"t2\", \"t3\"]}"
  },
  {
    "kb_version": 4,
    "change_type": "add_subflow",
    "payload": "{\"intent\": \"shipping_issue\", \"subflow\": \"missing\", \"tasks\": [\"s1\", \"s2\", \"s3\"]}"
  },
  {
    "kb_version": 5,
    "change_type": "attach_guideline",
    "payload": "{\"subflow\": \"reset_2fa\"}"
  },
  {
    "kb_version": 6,
    "change_type": "attach_guideline",
    "payload": "{\"subflow\": \"missing\"}"
  }
]
```

still unresolved intents: ['x1']
still unresolved subflows: ['l1', 'l2', 'l3']


### 20. Orchestrated parent invoke

Fresh `run_id` so LangSmith has one parent trace. The walkthrough already
mutated the bound KB, so this second invoke mostly classifies against the
updated structure.

In [15]:
orchestrated = invoke_week(
    "week-orchestrated",
    {
        "source": "scratch_data/incoming_conversations.json",
        "window": "demo-week",
        "as_of": "2026-09-01",
        "note": "second invoke against already-updated KB",
    },
    compiled=meta_graph,
)
show_state(orchestrated, title="orchestrated parent state")
show(store.get_staging("week-orchestrated", "run_summary"), "orchestrated run summary")
print("runtime kb_version", rt.playbook.version)

**orchestrated parent state**

```json
{
  "run_id": "week-orchestrated",
  "kb_version": 6,
  "current_stage": "summarize",
  "intent_summary": {
    "total": 12,
    "processed": 1,
    "classified": 0,
    "unknown": 1,
    "low_confidence": 0,
    "by_intent": {
      "account_access": 8,
      "shipping_issue": 3,
      "unknown": 1
    },
    "still_unresolved": 1
  },
  "subflow_summary": {
    "processed": 3,
    "classified": 0,
    "unknown": 3,
    "low_confidence": 0,
    "by_intent": {
      "account_access": {
        "unknown": 3,
        "recover_password": 1,
        "reset_2fa": 3,
        "recover_username": 1
      },
      "shipping_issue": {
        "missing": 3
      }
    },
    "still_unresolved": 3
  },
  "discovery_summary": {
    "intents": {
      "unresolved_tasks": 1,
      "candidate_count": 0,
      "outlier_count": 1,
      "approved_count": 0,
      "rejected_count": 1
    },
    "subflows": {
      "unresolved_tasks": 3,
      "candidate_count": 0,
      "emerging_count": 1,
      "approved_count": 0,
      "rejected_count": 1
    }
  },
  "recommendation_summary": {
    "items": [],
    "recommended": 0
  },
  "pending_proposal_ids": [
    "prop_45f740ea"
  ],
  "approved_change_ids": []
}
```

**orchestrated run summary**

```json
{
  "run_id": "week-orchestrated",
  "kb_version": 6,
  "cohort_size": 12,
  "intent_summary": {
    "total": 12,
    "processed": 1,
    "classified": 0,
    "unknown": 1,
    "low_confidence": 0,
    "by_intent": {
      "account_access": 8,
      "shipping_issue": 3,
      "unknown": 1
    },
    "still_unresolved": 1
  },
  "subflow_summary": {
    "processed": 3,
    "classified": 0,
    "unknown": 3,
    "low_confidence": 0,
    "by_intent": {
      "account_access": {
        "unknown": 3,
        "recover_password": 1,
        "reset_2fa": 3,
        "recover_username": 1
      },
      "shipping_issue": {
        "missing": 3
      }
    },
    "still_unresolved": 3
  },
  "discovery_summary": {
    "intents": {
      "unresolved_tasks": 1,
      "candidate_count": 0,
      "outlier_count": 1,
      "approved_count": 0,
      "rejected_count": 1
    },
    "subflows": {
      "unresolved_tasks": 3,
      "candidate_count": 0,
      "emerging_count": 1,
      "approved_count": 0,
      "rejected_count": 1
    }
  },
  "recommendation_summary": {
    "items": [],
    "recommended": 0
  },
  "intents_in_kb": [
    "account_access",
    "shipping_issue"
  ],
  "subflows_in_kb": {
    "account_access": [
      "recover_username",
      "recover_password",
      "reset_2fa"
    ],
    "shipping_issue": [
      "missing"
    ]
  },
  "unresolved_intents": [
    "x1"
  ],
  "unresolved_subflows": [
    "l1",
    "l2",
    "l3"
  ]
}
```

runtime kb_version 6


## LangSmith

Filter a parent run by tag `run:week-2026-09-01` or `run:week-orchestrated`.
Node metadata is `agent` + `phase`.

**Thread / session identity (not implemented here):** a later HITL phase can
map `run_id` to a LangGraph thread id and resume with `Command(resume=...)`.
This notebook still uses `simulate_hitl`.

In [16]:
try:
    from langsmith import Client

    client = Client()
    project = os.environ.get("LANGSMITH_PROJECT", "fresh-skills")
    runs = list(client.list_runs(project_name=project, is_root=True, limit=20))
    interesting = [
        run
        for run in runs
        if any(
            key in (run.name or "")
            for key in ("meta_agent", "classify", "discover", "recommend", "establish", "pathway", "intent", "subflow")
        )
    ]
    if interesting:
        display(Markdown("**Recent root LangSmith runs** (open a `meta_agent` / `classify_intents` parent and expand)"))
        for run in interesting[:8]:
            print(f"{run.name:40}  {getattr(run, 'url', '')}")
    else:
        print("No matching root runs yet. Re-run with LANGSMITH_TRACING=true and open the project in LangSmith.")
except Exception as exc:
    print("LangSmith lookup skipped:", type(exc).__name__, exc)

C:\Users\doste\AppData\Local\Temp\ipykernel_31616\2194032838.py:6: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  runs = list(client.list_runs(project_name=project, is_root=True, limit=20))


**Recent root LangSmith runs** (open a `meta_agent` / `classify_intents` parent and expand)

score_subflow_similarity                  https://smith.langchain.com/o/6d807a78-8d11-40a8-a318-5ff592ce77d8/projects/p/05454d08-51df-475d-9c21-bf18186abbf1/r/01a069b6-684b-7ba2-aa3e-b262e9b62b11?trace_id=01a069b6-684b-7ba2-aa3e-b262e9b62b11&start_time=2026-09-03T23:59:17.067645
score_subflow_similarity                  https://smith.langchain.com/o/6d807a78-8d11-40a8-a318-5ff592ce77d8/projects/p/05454d08-51df-475d-9c21-bf18186abbf1/r/01a069b6-6849-70f2-9369-d0bca9d02720?trace_id=01a069b6-6849-70f2-9369-d0bca9d02720&start_time=2026-09-03T23:59:17.064886
score_intent_similarity                   https://smith.langchain.com/o/6d807a78-8d11-40a8-a318-5ff592ce77d8/projects/p/05454d08-51df-475d-9c21-bf18186abbf1/r/01a069b6-673a-7b52-a5ce-9fd7b1dfcb0a?trace_id=01a069b6-673a-7b52-a5ce-9fd7b1dfcb0a&start_time=2026-09-03T23:59:16.793406
pathway:week-2026-09-01                   https://smith.langchain.com/o/6d807a78-8d11-40a8-a318-5ff592ce77d8/projects/p/05454d08-51df-475d-9c21-bf18186abbf1/r/0

## Notes for later

- Classification still uses Jaccard / LCS, not retrieval of similar labeled
  tasks. That swap is local to the classify process functions in `playbook.graph`.
- HITL is `simulate_hitl`. Real work is `interrupt()` + `Command(resume=...)`.
- Pathway drafts are still pass-through stubs.
- Do not copy this notebook over `demo_story.ipynb`; merge-back is `/apply-worktree`.